[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C74_Gaussian_Splatting_Course/04_densification/04_densification.ipynb)

# C74 · 模块 04 · 自适应密度控制

本 notebook 的核心是**把判据的可靠性量出来**，而不是复述它：

1. **3 个目标 × 5 个密度 × 2 个种子 = 每档 6 次独立运行**，
   算「累积平均位置梯度」与「局部残差」的秩相关与 top-1/4 命中率；
2. **两个替代判据（尺度梯度、不透明度梯度）的命中率都在随机基线附近或以下**；
3. **累积窗口从「全程」换成「最后 100 步」会把秩相关从 +0.68 打到 +0.43**；
4. 克隆 vs 分裂的体积恒等式：$2/\varphi^3$，$\varphi{=}1.6 \Rightarrow 0.488$；
5. 剪枝阈值的累积效应：**256 个 $\alpha{=}0.005$ 的高斯叠出 0.7229 的不透明度**；
6. 增长率的预算账：**要 20× 需要每轮 +22.1%**。

只用 numpy，CPU，离线。第 1 节约需 20 秒。

In [ ]:
import numpy as np
print('numpy', np.__version__)

X = np.linspace(0, 10, 2001)

def make_target(kind):
    '''三种目标，每种都含一个「容易拟合的平坦区」与一个「高频纹理区」。'''
    t = np.zeros_like(X)
    if kind == 'A':
        t += 0.6*((X > 1) & (X < 4))
        t += 0.5*(1 + np.sin(12*X))*((X > 5.5) & (X < 8.5))
    elif kind == 'B':
        t += 0.7*((X > 2) & (X < 3))
        t += 0.4*(1 + np.sin(20*X))*((X > 6) & (X < 9))
    else:
        t += 0.5*np.exp(-((X-2)**2)/0.3)
        t += 0.6*(1 + np.sin(8*X))*((X > 4.5) & (X < 9.5))
    return t

print('三个目标的形状（. 低 : 中 o 高 # 很高）：')
for kind in 'ABC':
    t = make_target(kind)
    line = ''.join('.' if v < 0.2 else (':' if v < 0.5 else ('o' if v < 0.8 else '#'))
                   for v in t[::25])
    print(f'  {kind}: {line}')
print('     x = 0' + ' '*68 + '10')

## 1 · 判据的可靠性：累积平均位置梯度 vs 局部残差

3DGS 的判据是 $\Vert \partial L/\partial \mu_{2D}\Vert_2$ 在最近 100 步上的平均。
这里用 1D 版本：累积 $\vert\partial L/\partial\mu\vert$ 的平均，
对比每个高斯「负责区域内的实际残差 RMS」（用它自己的高斯权重加权）。

In [ ]:
def fit_accum(K, tgt, iters=3000, lr=0.05, jitter=0.15, seed=0):
    '''K 个 1D 高斯拟合 tgt。返回三个累积平均梯度与每个高斯的局部残差 RMS。'''
    rng = np.random.default_rng(seed)
    mu = np.linspace(0.5, 9.5, K) + rng.normal(0, jitter, K)
    s = np.full(K, 9.0/K*0.5)
    a = np.full(K, 0.4)
    Am = np.zeros(K); As = np.zeros(K); Aa = np.zeros(K)
    for _ in range(iters):
        d = (X[None, :] - mu[:, None]) / s[:, None]
        g = np.exp(-0.5*d**2)
        r = (a[:, None]*g).sum(0) - tgt
        base = 2*r/len(X)
        ga = (base[None, :]*g).sum(1)
        gm = (base[None, :]*a[:, None]*g*(d/s[:, None])).sum(1)
        gs = (base[None, :]*a[:, None]*g*(d**2/s[:, None])).sum(1)
        Am += np.abs(gm); As += np.abs(gs); Aa += np.abs(ga)
        a -= lr*ga; mu -= lr*gm; s -= lr*gs
        s = np.clip(s, 1e-3, 5.0); a = np.clip(a, 0.0, 3.0)
    g = np.exp(-0.5*((X[None, :]-mu[:, None])/s[:, None])**2)
    r = (a[:, None]*g).sum(0) - tgt
    local = np.sqrt((g*r[None, :]**2).sum(1) / np.maximum(g.sum(1), 1e-12))
    return Am/iters, As/iters, Aa/iters, local

def rank(v):
    return np.argsort(np.argsort(v))

def spearman(u, v):
    return float(np.corrcoef(rank(u), rank(v))[0, 1])

def topk_hit(score, truth, k):
    '''score 最大的 k 个里，有几个也在 truth 最大的 k 个里（比例）。'''
    return len(set(np.argsort(score)[::-1][:k]) &
               set(np.argsort(truth)[::-1][:k])) / k

# 先看一个具体例子，把「梯度大的地方是不是残差大的地方」看清楚
Am, As, Aa, loc = fit_accum(16, make_target('A'), seed=0)
print('K=16, 目标 A。按累积位置梯度从大到小排：')
print(' 排名  |dL/dmu|      局部残差RMS   残差的排名')
res_rank = rank(loc)
for i, gi in enumerate(np.argsort(Am)[::-1]):
    print(f'  {i+1:2d}   {Am[gi]:.4e}   {loc[gi]:.4f}      {16-res_rank[gi]}')
print(f'\n秩相关 {spearman(Am, loc):+.3f}   top-4 命中 {topk_hit(Am, loc, 4):.0%}（基线 25%）')

In [ ]:
import time
t0 = time.time()
KS = [8, 12, 16, 24, 32]
res = {}
for K in KS:
    sp, h_mu, h_s, h_a = [], [], [], []
    for kind in 'ABC':
        tg = make_target(kind)
        for sd in range(2):
            Am, As, Aa, loc = fit_accum(K, tg, seed=sd)
            sp.append(spearman(Am, loc))
            k = max(2, K//4)
            h_mu.append(topk_hit(Am, loc, k))
            h_s.append(topk_hit(As, loc, k))
            h_a.append(topk_hit(Aa, loc, k))
    res[K] = dict(sp=np.array(sp), h_mu=np.mean(h_mu),
                  h_s=np.mean(h_s), h_a=np.mean(h_a), base=max(2, K//4)/K)
print(f'（{len(KS)} 档 × 3 目标 × 2 种子 = {len(KS)*6} 次运行，{time.time()-t0:.1f} s）\n')

print(' K    秩相关 均值    最小     最大    top-1/4 命中率            随机基线')
print('                                       |dL/dμ|  |dL/ds|  |dL/dα|')
for K in KS:
    r = res[K]
    print(f' {K:3d}    {r["sp"].mean():+.3f}      {r["sp"].min():+.3f}  {r["sp"].max():+.3f}'
          f'     {r["h_mu"]:5.0%}    {r["h_s"]:5.0%}   {r["h_a"]:5.0%}       {r["base"]:.0%}')

# ---- 结论 1：中等密度区判据有真实信号 ----
mid = [12, 16, 24]
for K in mid:
    assert res[K]['sp'].mean() > 0.55, f'K={K} 的秩相关均值应 >0.55，实测 {res[K]["sp"].mean():+.3f}'
    assert res[K]['h_mu'] > 1.6*res[K]['base'], \
        f'K={K} 的命中率应超基线 1.6 倍，实测 {res[K]["h_mu"]:.0%} vs {res[K]["base"]:.0%}'
print(f'\n✓ 结论 1：K=12/16/24 的秩相关 '
      f'{res[12]["sp"].mean():+.3f}/{res[16]["sp"].mean():+.3f}/{res[24]["sp"].mean():+.3f}，'
      f'命中率 {res[12]["h_mu"]:.0%}/{res[16]["h_mu"]:.0%}/{res[24]["h_mu"]:.0%}'
      f'（基线 25%）—— 判据有真实信号')

# ---- 结论 2：稀疏区判据 = 随机 ----
assert res[8]['sp'].min() < 0, f'K=8 应有运行给出负相关，实测最小 {res[8]["sp"].min():+.3f}'
assert abs(res[8]['h_mu'] - res[8]['base']) < 0.06, \
    f'K=8 的命中率应接近基线，实测 {res[8]["h_mu"]:.0%} vs {res[8]["base"]:.0%}'
print(f'✓ 结论 2：K=8 的命中率 {res[8]["h_mu"]:.0%} 恰等于随机基线 {res[8]["base"]:.0%}，'
      f'而某次运行的秩相关是 {res[8]["sp"].min():+.3f}')
print('  —— 稀疏时判据完全没有信息。这就是 densify_from_iter=500 的理由')

# ---- 结论 3：尺度梯度不是可用判据 ----
for K in KS:
    assert res[K]['h_s'] <= res[K]['base'] + 0.05, \
        f'K={K}: 尺度梯度命中率 {res[K]["h_s"]:.0%} 不该明显超过基线'
assert max(res[K]['h_s'] for K in mid) < min(res[K]['h_mu'] for K in mid), \
    '中等密度区，尺度梯度必须全面差于位置梯度'
print(f'✓ 结论 3：尺度梯度命中率在每一档都 ≤ 基线'
      f'（{"/".join(f"{res[K]["h_s"]:.0%}" for K in KS)}）—— 它不是可用判据')
print('  ⚠ 我第一次只跑了一个目标、一个种子，那次尺度梯度在 K=12 上命中 67%，')
print('    看起来比位置梯度好一倍。跑满 6 次取均值之后是 '
      f'{res[12]["h_s"]:.0%}。')
print('    「在单一配置上验证一个判据」这个陷阱，比结论本身更值得记住。')

# ---- 结论 4：不透明度梯度随密度变好 ----
assert res[8]['h_a'] < res[32]['h_a'] - 0.2, '不透明度梯度应随 K 显著变好'
print(f'✓ 结论 4：不透明度梯度从 K=8 的 {res[8]["h_a"]:.0%} 涨到 K=32 的 {res[32]["h_a"]:.0%}')
print('  高斯多了之后，「该不该存在」比「该往哪挪」更贴近残差')

### 1.1 累积窗口的长度直接改变结论

3DGS 在 **100 步**的窗口上平均。如果只用最后 100 步会怎样？如果只用最早 100 步？

In [ ]:
def fit_windowed(K, tgt, iters=3000, lr=0.05, jitter=0.15, seed=0,
                 win=(0, None)):
    '''同 fit_accum，但只在 [win[0], win[1]) 这段迭代里累积梯度。'''
    lo, hi = win[0], (win[1] if win[1] is not None else iters)
    rng = np.random.default_rng(seed)
    mu = np.linspace(0.5, 9.5, K) + rng.normal(0, jitter, K)
    s = np.full(K, 9.0/K*0.5); a = np.full(K, 0.4)
    Am = np.zeros(K); cnt = 0
    for it in range(iters):
        d = (X[None, :] - mu[:, None]) / s[:, None]
        g = np.exp(-0.5*d**2)
        r = (a[:, None]*g).sum(0) - tgt
        base = 2*r/len(X)
        ga = (base[None, :]*g).sum(1)
        gm = (base[None, :]*a[:, None]*g*(d/s[:, None])).sum(1)
        gs = (base[None, :]*a[:, None]*g*(d**2/s[:, None])).sum(1)
        if lo <= it < hi:
            Am += np.abs(gm); cnt += 1
        a -= lr*ga; mu -= lr*gm; s -= lr*gs
        s = np.clip(s, 1e-3, 5.0); a = np.clip(a, 0.0, 3.0)
    g = np.exp(-0.5*((X[None, :]-mu[:, None])/s[:, None])**2)
    r = (a[:, None]*g).sum(0) - tgt
    local = np.sqrt((g*r[None, :]**2).sum(1) / np.maximum(g.sum(1), 1e-12))
    return Am/cnt, local

print('K=16，三个目标 × 2 种子的秩相关：\n')
print('  累积窗口              均值      最小     最大    极差')
wins = {'全程 0~3000': (0, None), '前半 0~1500': (0, 1500),
        '后半 1500~3000': (1500, None), '中段 1000~1100': (1000, 1100),
        '最后 100 步': (2900, None), '最早 100 步': (0, 100)}
w_res = {}
for tag, w in wins.items():
    sp = []
    for kind in 'ABC':
        tg = make_target(kind)
        for sd in range(2):
            Am, loc = fit_windowed(16, tg, seed=sd, win=w)
            sp.append(spearman(Am, loc))
    w_res[tag] = np.array(sp)
    print(f'  {tag:20s}  {np.mean(sp):+.3f}   {min(sp):+.3f}  {max(sp):+.3f}'
          f'   {max(sp)-min(sp):.3f}')

_full = w_res['全程 0~3000']; _last = w_res['最后 100 步']; _first = w_res['最早 100 步']
# ① 窗口越长，均值越高
assert _full.mean() > _last.mean() + 0.10, \
    f'全程应好于最后 100 步：{_full.mean():+.3f} vs {_last.mean():+.3f}'
assert _last.mean() > _first.mean() + 0.10, \
    f'最后 100 步应好于最早 100 步：{_last.mean():+.3f} vs {_first.mean():+.3f}'
# ② 而更重要的效应是**方差**
_spread_full = _full.max() - _full.min()
_spread_last = _last.max() - _last.min()
assert _spread_last > 3*_spread_full, \
    f'短窗口的极差应是长窗口的 3 倍以上：{_spread_last:.3f} vs {_spread_full:.3f}'
# ③ 最早窗口会出现负相关的个例
assert _first.min() < 0.05, f'最早 100 步应出现接近零或负的个例，实测 {_first.min():+.3f}'

print(f'\n✓ 均值：全程 {_full.mean():+.3f} > 最后 100 步 {_last.mean():+.3f}'
      f' > 最早 100 步 {_first.mean():+.3f}')
print(f'✓ 而**更重要的效应是方差**：全程的极差 {_spread_full:.3f}，'
      f'最后 100 步 {_spread_last:.3f}（宽 {_spread_last/_spread_full:.1f} 倍）')
print()
print('三个结论，第二个纠正了一个常见的说法：')
print('  ① 窗口越长信号越强 —— 这是「在 100 步上平均而不是用单步」的理由')
print(f'  ② 最早 100 步**并非没有信号**（{_first.mean():+.3f}，约为全程的'
      f' {_first.mean()/_full.mean():.0%}），')
print(f'     但它有 1/6 的运行给出 {_first.min():+.3f} —— 也就是**不可靠**，而不是无信息。')
print('     densify_from_iter=500 的真实理由是「不可靠」，不是「没信号」')
print('  ③ 100 步窗口是一个折中：均值不如全程，方差还大 4 倍。')
print('     3DGS 之所以不用全程，是因为 densify 之后要**清零**重新积累 ——')
print('     它换来的是「能反映最近的状态」，代价就是这 4 倍的方差')

## 2 · 克隆 vs 分裂：一个体积恒等式

判据触发后选哪条路，只看「尺度是否超过场景尺度的 1%」。
两个操作对**总体积**的作用方向是**相反**的。

In [ ]:
def volume_ratio(phi, n_children=2):
    '''分裂成 n 个、每个尺度 /phi 之后，总体积相对原始的倍数。'''
    return n_children / phi**3

s0 = 0.10
print(f'原始各向同性高斯 scale = {s0} m，体积 ∝ s³ = {s0**3:.6f}\n')
print(f'克隆: scale 不变 {s0}，个数 ×2')
print(f'      总体积 {volume_ratio(1.0):.3f}×  <- 翻倍，「这里东西不够多」\n')
phi = 1.6
print(f'分裂: scale /{phi} = {s0/phi:.4f}，个数 ×2（父高斯被删除）')
print(f'      单个体积 {(s0/phi)**3:.6f} = 原始的 1/{phi**3:.2f}')
print(f'      总体积 {volume_ratio(phi):.4f}×  <- **缩小 {1-volume_ratio(phi):.1%}**，'
      f'「这里东西太粗，要细化」')

assert abs(volume_ratio(1.0) - 2.0) < 1e-12
assert abs(volume_ratio(1.6) - 0.48828125) < 1e-12, '2/1.6³ 必须是 0.48828125'

print(f'\nφ 的临界值：2/φ³ = 1 <=> φ = 2^(1/3) = {2**(1/3):.4f}')
print('  φ         总体积倍数')
for p in [1.0, 1.2, 2**(1/3), 1.4, 1.6, 2.0, 2.5]:
    v = volume_ratio(p)
    tag = '不变' if abs(v-1) < 1e-9 else ('放大' if v > 1 else '缩小')
    print(f'  {p:.4f}     {v:.4f}   {tag}')
assert abs(volume_ratio(2**(1/3)) - 1.0) < 1e-12, '临界点必须恰好体积不变'
print(f'\n✓ 官方选 φ=1.6 > {2**(1/3):.4f}，所以分裂是**明确地缩小**')
print('  推论：分裂真的改变了渲染结果（模块 01 第 7 节：3DGS 没有步长概念），')
print('        所以分裂之后必须继续优化才能收敛回去 —— 它不是等价变换')

# 位置从父高斯自身的分布里采样 -> 方向信息免费继承
print('\n「位置从父高斯的分布里采样」这个细节：')
rng = np.random.default_rng(0)
for scale, tag in [([0.10, 0.10, 0.10], '各向同性'),
                   ([0.30, 0.03, 0.03], '沿 x 的针'),
                   ([0.30, 0.30, 0.01], '薄片')]:
    S = np.diag(np.array(scale)**2)
    kids = rng.multivariate_normal(np.zeros(3), S, 4000)
    span = kids.std(0)
    print(f'  {tag:8s} scale={scale}: 子高斯位置的标准差 {np.round(span,4)}')
    assert np.argmax(span) == np.argmax(scale), '子高斯必须沿父高斯的长轴分布'
print('  ✓ 子高斯自动沿父高斯的长轴分布 —— 方向信息从协方差里免费继承，无需额外判断')
print('  代价：这是**随机**的。同一个高斯分裂两次结果不同 -> 3DGS 训练不是确定性的')

## 3 · 剪枝阈值：为什么不能只看单个高斯

In [ ]:
def accumulated_alpha(alpha, n):
    '''n 个不透明度均为 alpha 的高斯叠加后的累积不透明度。'''
    return 1.0 - (1.0 - alpha)**n

def n_for_target(alpha, target):
    '''达到 target 累积不透明度所需的高斯个数。'''
    return int(np.ceil(np.log(1-target)/np.log(1-alpha)))

print('单个 α      1 个      16 个     64 个     256 个')
for al in [0.005, 0.010, 0.050]:
    row = '  '.join(f'{accumulated_alpha(al, n):.4f}' for n in [1, 16, 64, 256])
    print(f'  {al:.3f}    {row}')

a256 = accumulated_alpha(0.005, 256)
assert abs(a256 - 0.7229) < 1e-4, f'256 个 α=0.005 应叠出 0.7229，实测 {a256:.4f}'
print(f'\n✓ 256 个 α=0.005 的高斯叠出 {a256:.4f} 的不透明度')
print('  而模块 03 量过：1920x1080 下一个 tile 里平均就有 245 个高斯')
print('  所以「几乎透明」的高斯堆在一起完全可以是不透明的 ——')
print('  0.005 这个阈值不能从「单个高斯的可见性」推出来，它是经验值')

print(f'\n达到 50% / 90% 不透明度需要多少个：')
for al in [0.005, 0.01, 0.05, 0.1]:
    print(f'  α={al:.3f}: 50% 需 {n_for_target(al,0.5):4d} 个，'
          f'90% 需 {n_for_target(al,0.9):4d} 个')
assert n_for_target(0.005, 0.5) == 139
# 反过来：阈值定在哪，才能保证 245 个叠起来也看不见（<1/255）？
print(f'\n反推：要让 245 个叠起来仍低于 1/255 = {1/255:.5f}，单个 α 上限是多少？')
lo, hi = 1e-9, 0.01
for _ in range(200):
    mid = (lo+hi)/2
    if accumulated_alpha(mid, 245) < 1/255:
        lo = mid
    else:
        hi = mid
print(f'  α < {lo:.2e}  —— 比官方的 0.005 小了 {0.005/lo:.0f} 倍')
assert lo < 5e-5
print('  ✓ 所以官方的 0.005 **不是**按「叠起来也看不见」定的。')
print('    它的真实含义更接近「这个高斯已经不再被优化器有效使用」')

## 4 · 增长率的预算账

In [ ]:
def growth_rate(n_init, n_final, rounds=15):
    '''达到目标倍数所需的每轮复合增长率。'''
    return (n_final/n_init)**(1.0/rounds) - 1.0

print('每轮增长    15 轮后     从 10 万长到')
for p in [0.05, 0.10, 0.15, 0.221, 0.298]:
    print(f'  +{p:.1%}      {(1+p)**15:6.2f}×      {100_000*(1+p)**15/1e4:6.1f} 万')

print('\n反推：')
for target in [5, 10, 20, 50]:
    p = growth_rate(1, target, 15)
    print(f'  想要 {target:2d}× 需要每轮 +{p:.1%}')
p20 = growth_rate(100_000, 2_000_000, 15)
assert abs(p20 - 0.221) < 5e-4, f'10 万 -> 200 万需每轮 +22.1%，实测 {p20:.1%}'
print(f'\n✓ 10 万 -> 200 万（20×）需要每轮 +{p20:.1%}')

# 指数敏感性
print('\n阈值的指数敏感性：')
for delta in [0.8, 0.9, 1.0, 1.1, 1.25]:
    p = 0.221*delta
    print(f'  每轮增长变成基准的 {delta:.2f}× (+{p:.1%}): 终值 {(1+p)**15:6.2f}×'
          f'  相对基准 {(1+p)**15/(1.221**15):5.2f}×')
r_hi = (1+0.221*1.25)**15 / 1.221**15
assert r_hi > 1.9, f'增长率涨 25% 时终值应涨近 2 倍，实测 {r_hi:.2f}×'
print(f'\n✓ 每轮增长率涨 25%（+22.1% -> +27.6%），最终高斯数涨 {r_hi:.2f}×')
print(f'  而涨到 +29.8% 时是 {(1.298**15)/(1.221**15):.2f}× —— 复合增长是指数敏感的')
print('  所以密度阈值降 20% 就可能让最终数量翻倍进而 OOM，')
print('  而这正是后续工作改成「直接给预算 N_max」的理由')

# 显存账
print('\n显存账（SH 阶 3）：')
per_g = 3+3+4+1+48
for name, mult in [('只存参数', 1), ('参数 + Adam 一二阶动量', 3)]:
    for gb in [8, 12, 24]:
        n = gb*1e9/(per_g*4*mult)
        print(f'  {name:22s} {gb:2d} GB -> {n/1e6:5.1f} M 个高斯')
print('  但训练时还要放梯度、渲染缓冲与 (高斯,tile) 对（模块 03：2.00M 个）')
print('  所以 12 GB 的实际上限大约在 300-500 万，而不是 1700 万')

---
## ✏️ 练习

四道题各自独立。先写 TODO，再跑下一格的自测。

### ✏️ 练习 1 · 累积平均位置梯度

实现 `my_accum(K, tgt, iters, lr, jitter, seed)`，返回 `(accum_mu, local_res)`：
`K` 个 1D 高斯拟合 `tgt`，返回**累积平均** $\vert\partial L/\partial\mu\vert$
与每个高斯的局部残差 RMS（用它自己的高斯权重加权）。

初始化与解析梯度都照 `fit_accum` 的写法（损失是 `mean((f-tgt)²)`）。

In [ ]:
def my_accum(K, tgt, iters=3000, lr=0.05, jitter=0.15, seed=0):
    '''返回 (累积平均 |dL/dmu| (K,), 局部残差 RMS (K,))。'''
    # TODO: rng = np.random.default_rng(seed)
    #   mu = linspace(0.5,9.5,K) + rng.normal(0,jitter,K); s = full(K, 9/K*0.5); a = full(K,0.4)
    #   每步：d=(X-mu)/s; g=exp(-d²/2); r=(a*g).sum(0)-tgt; base=2r/len(X)
    #         ga=(base*g).sum(1); gm=(base*a*g*(d/s)).sum(1); gs=(base*a*g*(d²/s)).sum(1)
    #         累加 |gm|；再 a-=lr*ga; mu-=lr*gm; s-=lr*gs; clip s 到 [1e-3,5], a 到 [0,3]
    #   末尾：g 重算；r=(a*g).sum(0)-tgt
    #         local = sqrt((g*r²).sum(1)/max(g.sum(1),1e-12))
    #   返回 (累加/iters, local)
    raise NotImplementedError

In [ ]:
# ---- 自测 1 ----
_tgA, _tgB = make_target('A'), make_target('B')

# ① 形状与非负
_am, _lc = my_accum(16, _tgA, iters=1500, seed=0)
assert _am.shape == (16,) and _lc.shape == (16,), f'形状 {_am.shape} {_lc.shape}'
assert np.all(_am >= 0), '累积的是绝对值，必须非负'
assert np.all(_lc >= 0)

# ② 与参考实现一致（同样的 iters/seed）
_ref_m, _ref_s, _ref_a, _ref_l = fit_accum(16, _tgA, iters=1500, seed=0)
assert np.allclose(_am, _ref_m, rtol=1e-9), '累积梯度与参考不符'
assert np.allclose(_lc, _ref_l, rtol=1e-9), '局部残差与参考不符'

# ③ 中等密度区必须有真实信号（3 组配置）
_sp = []
for _K in [12, 16, 24]:
    for _tg in [_tgA, _tgB]:
        _a2, _l2 = my_accum(_K, _tg, iters=1500, seed=0)
        _sp.append(spearman(_a2, _l2))
assert np.mean(_sp) > 0.45, f'中等密度区秩相关均值应 >0.45，实测 {np.mean(_sp):+.3f}'
assert min(_sp) > 0.0, f'中等密度区不该出现负相关，实测最小 {min(_sp):+.3f}'

# ④ 稀疏区（K=8）必须明显更差
_sp8 = [spearman(*my_accum(8, _tg, iters=1500, seed=_sd))
        for _tg in [_tgA, _tgB] for _sd in range(2)]
assert np.mean(_sp8) < np.mean(_sp) - 0.15, \
    f'K=8 应明显差于中等密度区：{np.mean(_sp8):+.3f} vs {np.mean(_sp):+.3f}'

# ⑤ 确定性：同 seed 必须逐位可复现
assert np.array_equal(my_accum(12, _tgA, iters=800, seed=1)[0],
                      my_accum(12, _tgA, iters=800, seed=1)[0]), '同 seed 必须可复现'
print(f'✓ 练习 1 通过：中等密度区秩相关均值 {np.mean(_sp):+.3f}（最小 {min(_sp):+.3f}），'
      f'K=8 只有 {np.mean(_sp8):+.3f}')

### 📖 参考答案 1

In [ ]:
def my_accum(K, tgt, iters=3000, lr=0.05, jitter=0.15, seed=0):
    rng = np.random.default_rng(seed)
    mu = np.linspace(0.5, 9.5, K) + rng.normal(0, jitter, K)
    s = np.full(K, 9.0/K*0.5)
    a = np.full(K, 0.4)
    Am = np.zeros(K)
    for _ in range(iters):
        d = (X[None, :] - mu[:, None]) / s[:, None]
        g = np.exp(-0.5*d**2)
        r = (a[:, None]*g).sum(0) - tgt
        base = 2*r/len(X)
        ga = (base[None, :]*g).sum(1)
        gm = (base[None, :]*a[:, None]*g*(d/s[:, None])).sum(1)
        gs = (base[None, :]*a[:, None]*g*(d**2/s[:, None])).sum(1)
        Am += np.abs(gm)
        a -= lr*ga; mu -= lr*gm; s -= lr*gs
        s = np.clip(s, 1e-3, 5.0); a = np.clip(a, 0.0, 3.0)
    g = np.exp(-0.5*((X[None, :]-mu[:, None])/s[:, None])**2)
    r = (a[:, None]*g).sum(0) - tgt
    local = np.sqrt((g*r[None, :]**2).sum(1) / np.maximum(g.sum(1), 1e-12))
    return Am/iters, local

print('参考答案 1 已定义')
print('要点一：累加的是 |gm| 而不是 gm。累加带符号的梯度会互相抵消 ——')
print('       而「被不同方向拉扯」正是判据想抓的信号，抵消掉就什么都不剩了。')
print('       3DGS 里对应的是 ‖∂L/∂μ_2D‖₂（2D 向量的模长），道理相同。')
print('要点二：局部残差用**高斯自己的权重**加权，而不是固定窗口 ——')
print('       因为「这个高斯负责哪一片」本身就由它的 σ 决定。')
print('要点三：这个判据在 K=8 时的秩相关接近 0（甚至为负）。')
print('       所以它不是定理，是启发式 —— densify_from_iter=500 就是这个原因。')

### ✏️ 练习 2 · 克隆与分裂的体积恒等式

实现 `my_volume_ratio(phi, n_children)`：分裂成 `n_children` 个、
每个尺度除以 `phi` 之后，**总体积**相对原始的倍数。
再实现 `my_critical_phi(n_children)`：使总体积不变的临界 `phi`。

In [ ]:
def my_volume_ratio(phi, n_children=2):
    '''总体积倍数。'''
    # TODO: 体积 ∝ s³，所以单个变 1/phi³，n 个合计 n/phi³
    raise NotImplementedError

def my_critical_phi(n_children=2):
    '''使总体积不变的 phi。'''
    # TODO: 解 n/phi³ = 1
    raise NotImplementedError

In [ ]:
# ---- 自测 2 ----
# ① 克隆（phi=1）总体积翻倍
assert abs(my_volume_ratio(1.0, 2) - 2.0) < 1e-12
# ② 官方的 phi=1.6
assert abs(my_volume_ratio(1.6, 2) - 0.48828125) < 1e-12, \
    f'2/1.6³ 必须是 0.48828125，实测 {my_volume_ratio(1.6,2)}'
# ③ 临界点
_pc = my_critical_phi(2)
assert abs(_pc - 2**(1/3)) < 1e-12, f'临界 phi 应为 2^(1/3)={2**(1/3):.6f}，实测 {_pc}'
assert abs(my_volume_ratio(_pc, 2) - 1.0) < 1e-12, '临界点处体积必须恰好不变'
# ④ 单调性：phi 越大总体积越小
_vs = [my_volume_ratio(p, 2) for p in [1.0, 1.2, 1.4, 1.6, 2.0, 2.5]]
assert all(_vs[i] > _vs[i+1] for i in range(len(_vs)-1)), '必须单调递减'
# ⑤ 三个子高斯的情形
assert abs(my_volume_ratio(1.6, 3) - 3/1.6**3) < 1e-12
assert abs(my_critical_phi(3) - 3**(1/3)) < 1e-12
assert my_critical_phi(3) > my_critical_phi(2), '子高斯越多，临界 phi 越大'
# ⑥ 与参考实现一致
for _p in [1.0, 1.2599, 1.6, 2.0]:
    assert abs(my_volume_ratio(_p, 2) - volume_ratio(_p, 2)) < 1e-12
print(f'✓ 练习 2 通过：φ=1.6 -> {my_volume_ratio(1.6):.5f}×（缩小 '
      f'{1-my_volume_ratio(1.6):.1%}）；临界 φ = {_pc:.4f}；'
      f'n=3 的临界 φ = {my_critical_phi(3):.4f}')

### 📖 参考答案 2

In [ ]:
def my_volume_ratio(phi, n_children=2):
    return n_children / phi**3

def my_critical_phi(n_children=2):
    return n_children**(1.0/3.0)

print('参考答案 2 已定义')
print('要点：官方的 φ=1.6 明显大于临界值 1.2599，所以分裂是**确定地缩小总体积**。')
print('     这不是副作用，这就是「细化」的机制本身：')
print('       克隆说「这里东西不够多」（体积 ×2），')
print('       分裂说「这里东西太粗」（体积 ×0.488）。')
print('     而因为 3DGS 没有步长概念（模块 01 第 7 节），分裂真的改变了渲染结果，')
print('     所以分裂之后必须继续优化 —— 它不是一个等价变换。')

### ✏️ 练习 3 · 剪枝阈值的累积效应

实现两个互逆的函数：
`my_acc_alpha(alpha, n)` = $n$ 个不透明度 `alpha` 的高斯叠加后的累积不透明度；
`my_alpha_for(n, target)` = 使 $n$ 个叠出 `target` 累积不透明度所需的单个 `alpha`。

In [ ]:
def my_acc_alpha(alpha, n):
    '''n 个 α 相同的高斯叠加后的累积不透明度。'''
    # TODO: 1 - (1-alpha)^n
    raise NotImplementedError

def my_alpha_for(n, target):
    '''使 n 个叠出 target 所需的单个 alpha（解析解，不要二分）。'''
    # TODO: 由 1-(1-α)^n = target 解得 α = 1 - (1-target)^(1/n)
    raise NotImplementedError

In [ ]:
# ---- 自测 3 ----
# ① 官方阈值的累积效应
_a = my_acc_alpha(0.005, 256)
assert abs(_a - 0.7229) < 1e-4, f'256 个 α=0.005 应叠出 0.7229，实测 {_a:.4f}'
assert abs(my_acc_alpha(0.005, 1) - 0.005) < 1e-15, 'n=1 时应等于 alpha 本身'
# ② 单调性
assert all(my_acc_alpha(0.005, n) < my_acc_alpha(0.005, n+1) for n in [1, 16, 64, 255])
assert all(my_acc_alpha(a, 64) < my_acc_alpha(a+0.001, 64) for a in [0.005, 0.01, 0.05])
# ③ 与参考实现一致
for _al in [0.005, 0.01, 0.05, 0.5]:
    for _n in [1, 16, 245, 1000]:
        assert abs(my_acc_alpha(_al, _n) - accumulated_alpha(_al, _n)) < 1e-14
# ④ 互逆
for _n in [1, 16, 245, 1000]:
    for _t in [0.01, 0.5, 0.9, 0.99]:
        _al = my_alpha_for(_n, _t)
        assert 0 < _al < 1, f'n={_n} target={_t}: alpha={_al} 越界'
        assert abs(my_acc_alpha(_al, _n) - _t) < 1e-12, \
            f'必须互逆: n={_n} target={_t} -> alpha={_al} -> {my_acc_alpha(_al,_n)}'
# ⑤ 反推那个关键结论：要让 245 个叠起来仍 < 1/255，单个 alpha 上限
_cap = my_alpha_for(245, 1/255)
assert _cap < 5e-5, f'上限应 <5e-5，实测 {_cap:.2e}'
assert 0.005/_cap > 100, f'官方的 0.005 应比它大 100 倍以上，实测 {0.005/_cap:.0f}×'
# ⑥ 边界
assert abs(my_alpha_for(1, 0.5) - 0.5) < 1e-15
# α=0.999 的 100 个叠起来：(1-α)^n = 1e-300 在 float64 下仍可表示，但 1-1e-300 == 1.0
assert my_acc_alpha(0.999, 100) == 1.0, '应因浮点吸收而恰好等于 1.0'
assert (1-0.999)**100 > 0, '而 (1-α)^n 本身还没下溢'
print(f'✓ 练习 3 通过：256 个 α=0.005 叠出 {_a:.4f}；'
      f'要让 245 个仍 <1/255 需 α < {_cap:.2e}，'
      f'而官方阈值 0.005 是它的 {0.005/_cap:.0f} 倍')
print(f'  顺带一个浮点细节：α=0.999 的 100 个叠起来，(1-α)^n = {(1-0.999)**100:.1e} '
      f'还没下溢，\n  但 1-{(1-0.999)**100:.1e} 在 float64 下**恰好等于 1.0** —— '
      f'与模块 01 里 T 的下溢是同一类问题')

### 📖 参考答案 3

In [ ]:
def my_acc_alpha(alpha, n):
    return 1.0 - (1.0 - alpha)**n

def my_alpha_for(n, target):
    return 1.0 - (1.0 - target)**(1.0/n)

print('参考答案 3 已定义')
print('要点：⑤ 那个反推是本节的重点。')
print('     如果 0.005 这个阈值真的是按「叠起来也看不见」定的，它应该是 2e-5 量级；')
print('     它比那个值大 100 多倍，说明它的依据**不是**可见性。')
print('     更贴近的解释是「α 已经小到优化器不再有效使用它」——')
print('     而这也解释了为什么它要和不透明度重置配对使用（讲解页第 5 节）：')
print('     重置把所有 α 压到 0.01，真正有用的会长回来，浮物不会，')
print('     然后 α<0.005 这条规则把它们清掉。单独任何一个机制都不起作用。')

### ✏️ 练习 4 · 增长率的预算账

实现 `my_growth(n_init, n_final, rounds)`：达到目标所需的**每轮复合增长率**；
以及 `my_final(n_init, p, rounds)`：给定增长率算终值。

In [ ]:
def my_growth(n_init, n_final, rounds=15):
    '''每轮复合增长率 p，使 n_init*(1+p)^rounds = n_final。'''
    # TODO: (n_final/n_init)^(1/rounds) - 1
    raise NotImplementedError

def my_final(n_init, p, rounds=15):
    '''给定每轮增长率算终值。'''
    # TODO: n_init * (1+p)^rounds
    raise NotImplementedError

In [ ]:
# ---- 自测 4 ----
# ① 官方场景：10 万 -> 200 万
_p = my_growth(100_000, 2_000_000, 15)
assert abs(_p - 0.221) < 5e-4, f'应为 +22.1%，实测 {_p:.4%}'
# ② 互逆
for _ni, _nf, _r in [(1e5, 5e5, 15), (1e5, 5e6, 15), (1e5, 2e6, 30), (1000, 1000, 15)]:
    _pp = my_growth(_ni, _nf, _r)
    assert abs(my_final(_ni, _pp, _r) - _nf)/_nf < 1e-12, '必须互逆'
# ③ 不增长时 p=0
assert abs(my_growth(1e5, 1e5, 15)) < 1e-15
# ④ 与参考实现一致
for _t in [5, 10, 20, 50]:
    assert abs(my_growth(1, _t, 15) - growth_rate(1, _t, 15)) < 1e-12
# ⑤ 指数敏感性：增长率涨 25%，终值必须涨 2 倍以上
_base = my_final(1, 0.221, 15)
_hi = my_final(1, 0.221*1.25, 15)
assert _hi/_base > 1.9, f'增长率涨 25% 时终值应涨近 2×，实测 {_hi/_base:.2f}×'
# ⑥ 轮数越多，达到同一目标所需的每轮增长越小
assert my_growth(1e5, 2e6, 15) > my_growth(1e5, 2e6, 30) > my_growth(1e5, 2e6, 100)
# ⑦ 反向：显存预算倒推
_per_g_bytes = (3+3+4+1+48)*4
_n_cap = 12e9/(_per_g_bytes*3)             # 参数 + Adam 动量
_p_cap = my_growth(100_000, _n_cap, 15)
assert 0.30 < _p_cap < 0.45, f'12GB 的理论上限对应每轮 +{_p_cap:.1%}'
print(f'✓ 练习 4 通过：10 万->200 万需每轮 +{_p:.1%}；'
      f'增长率涨 25% 终值涨 {_hi/_base:.2f}×；'
      f'12GB 理论上限 {_n_cap/1e6:.1f}M 个对应每轮 +{_p_cap:.1%}')

### 📖 参考答案 4

In [ ]:
def my_growth(n_init, n_final, rounds=15):
    return (n_final/n_init)**(1.0/rounds) - 1.0

def my_final(n_init, p, rounds=15):
    return n_init * (1.0 + p)**rounds

print('参考答案 4 已定义')
print('要点：⑤ 的指数敏感性是本节最有用的一条 ——')
print('     每轮增长率只涨 25%，最终高斯数涨 2 倍以上。')
print('     所以「把密度阈值从 2e-4 调到 1.6e-4」这种看起来温和的改动，')
print('     可能直接把你从「刚好放得下」推到 OOM。')
print()
print('     而 ⑦ 说明另一件事：12GB 的**理论**上限（1.7M 个）对应每轮 +38%，')
print('     远高于官方的 +22%。所以官方的默认值是保守的 ——')
print('     因为实际还要放梯度、渲染缓冲与 200 万个 (高斯,tile) 对（模块 03）。')

---
## 🧪 真实工程胶囊

```python
# ---- 官方实现（scene/gaussian_model.py）----
def densify_and_prune(self, max_grad, min_opacity, extent, max_screen_size):
    grads = self.xyz_gradient_accum / self.denom       # ← 练习 1 的「累积平均」
    grads[grads.isnan()] = 0.0
    self.densify_and_clone(grads, max_grad, extent)
    self.densify_and_split(grads, max_grad, extent)
    prune_mask = (self.get_opacity < min_opacity).squeeze()   # min_opacity = 0.005
    if max_screen_size:                                        # = 20 px
        big_points_vs = self.max_radii2D > max_screen_size
        big_points_ws = self.get_scaling.max(dim=1).values > 0.1 * extent
        prune_mask = torch.logical_or(torch.logical_or(prune_mask, big_points_vs),
                                      big_points_ws)
    self.prune_points(prune_mask)

def add_densification_stats(self, viewspace_point_tensor, update_filter):
    # ← 判据就是这一行：屏幕空间位置梯度的 L2 模长，逐次累加
    self.xyz_gradient_accum[update_filter] += torch.norm(
        viewspace_point_tensor.grad[update_filter, :2], dim=-1, keepdim=True)
    self.denom[update_filter] += 1

def densify_and_clone(self, grads, grad_threshold, scene_extent):
    selected = torch.logical_and(
        torch.norm(grads, dim=-1) >= grad_threshold,
        torch.max(self.get_scaling, dim=1).values <= self.percent_dense*scene_extent)
    # 尺度**不变**，直接复制 -> 总体积 ×2（练习 2）
    self.densification_postfix(self._xyz[selected], ..., self._scaling[selected], ...)

def densify_and_split(self, grads, grad_threshold, scene_extent, N=2):
    selected = torch.logical_and(
        padded_grad >= grad_threshold,
        torch.max(self.get_scaling, dim=1).values > self.percent_dense*scene_extent)
    stds = self.get_scaling[selected].repeat(N, 1)
    means = torch.zeros((stds.size(0), 3), device="cuda")
    samples = torch.normal(mean=means, std=stds)      # ← 从父高斯的分布里采样
    rots = build_rotation(self._rotation[selected]).repeat(N, 1, 1)
    new_xyz = torch.bmm(rots, samples.unsqueeze(-1)).squeeze(-1) + self.get_xyz[selected]
    new_scaling = self.scaling_inverse_activation(
        self.get_scaling[selected].repeat(N,1) / (0.8*N))   # ← 0.8*N = 1.6，练习 2
    ...
    self.prune_points(prune_filter)                    # 父高斯被删除

# 关键默认值（arguments/__init__.py）
#   densify_from_iter = 500        densify_until_iter = 15_000
#   densification_interval = 100   densify_grad_threshold = 0.0002
#   opacity_reset_interval = 3000  percent_dense = 0.01

# ---- 改判据的三种做法 ----
# Pixel-GS: 把 add_densification_stats 里的累加改成按覆盖像素数加权
# Mini-Splatting: 换掉整套 clone/split，改成「先长大，再按重要度采样压回预算」
# 3DGS-MCMC: pip install gsplat 后用 strategy=MCMCStrategy(cap_max=1_000_000)
#            总数固定，clone/split/prune 换成「重定位 + 噪声注入」
```

**排查清单**

| 症状 | 先查什么 | 依据 |
|---|---|---|
| 高斯数几乎不长 | 打印每轮触发的**个数**；`densify_from_iter` | 判据在早期几乎无信号（秩相关 −0.16） |
| 高斯数爆炸 OOM | 阈值；把总数画成对数曲线看是否为直线 | 增长率涨 25% → 终值涨 2 倍以上 |
| 细节永远糊但数量在长 | 新增高斯的**空间分布** | 判据 top-1/4 命中率只有 44–56% |
| 悬空的半透明色块 | `opacity_reset_interval`；从训练集外的视角渲一张 | 浮物的位置梯度小，判据看不见 |
| PSNR 周期性尖峰下跌 | **正常** —— 就是不透明度重置 | 周期应等于 3000 |
| 重置后 PSNR 不恢复 | 重置是否发生在 `densify_until_iter` 之后 | 恢复需要密度控制还在跑 |
| 换数据集就要重调阈值 | **正常** —— 判据的标度不稳定 | 同一 $K$ 换目标，秩相关 +0.90 → −0.26 |